# Similarité Mayence 1545 → Gallica

**Projet** : Gallica Images — Illustrations des *Métamorphoses* d'Ovide  
**Date**   : Avril 2026

Ce notebook calcule les embeddings CLIP des illustrations segmentées de l'édition
de Mayence 1545 (Jörg Wickram) via l'endpoint `/api/image` de l'API BnF,
puis recherche pour chacune les illustrations les plus similaires dans Gallica.

**Entrée**  : `data/segmentees/bois_wickram_mayence1545/`  
**Sortie**  : `resultats/csv/mayence1545.csv`  
             `resultats/similarite/correspondances_mayence1545_gallica.pdf`

---

**Pipeline :**
1. Pour chaque illustration segmentée — envoi à `POST /api/image` → vecteur CLIP 768D
2. Envoi du vecteur à `POST /api/search` avec `type="image"` → 10 résultats Gallica
3. Sauvegarde en CSV
4. Export PDF — une page par illustration avec le meilleur résultat Gallica

---

## 1. Configuration

In [1]:
import requests, json, os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
from io import BytesIO

BASE_URL        = "https://gallica-search-api-preprod.bnf.lajavaness.com"
DOSSIER_MAYENCE = "../../data/segmentees/bois_wickram_mayence1545"
CHEMIN_CSV      = "../../resultats/csv/mayence1545.csv"
CHEMIN_PDF      = "../../resultats/similarite/correspondances_mayence1545_gallica.pdf"
N_RESULTATS     = 10

def embedding_depuis_image(img_bytes):
    """
    Envoie une image à l'endpoint /api/image de l'API BnF
    et retourne son vecteur CLIP 768D.
    """
    r = requests.post(
        f"{BASE_URL}/api/image",
        files={"image": ("image.jpg", img_bytes, "image/jpeg")},
        timeout=30
    )
    if r.status_code == 200:
        return json.loads(r.text)
    return None

def rechercher_similaires(vecteur, n=10):
    """Recherche les n illustrations les plus similaires dans Gallica."""
    r = requests.post(
        f"{BASE_URL}/api/search",
        json={"type": "image", "image_vector": vecteur, "rows": n, "start": 0},
        headers={"Content-Type": "application/json"},
        timeout=30
    )
    return r.json()["response"]["docs"]

def get_image_url(url):
    """Télécharge une image depuis une URL Gallica."""
    try:
        r = requests.get(url, timeout=15)
        return Image.open(BytesIO(r.content)).convert("RGB")
    except:
        return None

images = [f for f in os.listdir(DOSSIER_MAYENCE)
          if f.endswith(".jpg") and "_flip" not in f]
print(f"✓ {len(images)} illustrations à traiter")

✓ 52 illustrations à traiter


## 2. Calcul des embeddings et recherche dans Gallica

Pour chaque illustration segmentée : envoi à `/api/image` → vecteur CLIP → `/api/search` → 10 résultats.

In [2]:
resultats = []

for i, nom in enumerate(images):
    print(f"  {i+1}/{len(images)}...", end="\r")
    chemin = f"{DOSSIER_MAYENCE}/{nom}"
    try:
        with open(chemin, "rb") as f:
            img_bytes = f.read()

        vecteur = embedding_depuis_image(img_bytes)
        if vecteur is None:
            continue

        docs = rechercher_similaires(vecteur, N_RESULTATS)

        for doc in docs:
            resultats.append({
                "mayence_image" : nom,
                "score"         : round(doc.get("score", 0), 4),
                "titre"         : (doc.get("context_title")  or [""])[0][:80],
                "auteur"        : (doc.get("context_author") or [""])[0][:50],
                "date"          : str(doc.get("context_date", ""))[:4],
                "corpus"        : (doc.get("context_corpus") or [""])[0][:60],
                "technique"     : (doc.get("properties_technical_category") or [""])[0],
                "genre"         : (doc.get("properties_genre")               or [""])[0],
                "link"          : doc.get("link", ""),
            })
    except Exception as e:
        print(f"\n  Erreur {nom} : {e}")

df_mayence = pd.DataFrame(resultats)
df_mayence.to_csv(CHEMIN_CSV, index=False)
print(f"\n✓ {len(df_mayence)} résultats pour {len(images)} illustrations")
print(f"✓ CSV sauvegardé : {CHEMIN_CSV}")

  52/52...
✓ 520 résultats pour 52 illustrations
✓ CSV sauvegardé : ../../resultats/similarite_mayence1545.csv


## 3. Statistiques générales

In [3]:
print("Répartition par technique :")
print(df_mayence["technique"].value_counts().to_string())
print()
print("Distribution des scores :")
print(df_mayence["score"].describe().round(3).to_string())
print()
print("Top 10 titres les plus fréquents :")
print(df_mayence["titre"].value_counts().head(10).to_string())

Répartition par technique :
technique
estampe         395
photographie    124
dessin            1

Distribution des scores :
count    520.000
mean       0.941
std        0.009
min        0.914
25%        0.935
50%        0.942
75%        0.946
max        0.973

Top 10 titres les plus fréquents :
titre
[Illustrations de Les Métamorphoses] / Georges Wickram, grav. ; Ovide, aut. du t    108
emblemata, elucidata doctissimis Claudii Minois commentariis, quibus additae sun     42
[Illustrations de Histoire des pays septentrionaux] / [Non identifié] ; Olus Le      37
[Horae ad usum Cabilonensem. Lat. et fr.]                                            25
Los quatro libros de Amadis de Gaula nueuamente impressos & hystoriados en Sevil     23
[Illustrations de Les Métamorphoses] / [Non identifié] ; Ovide, aut. du texte        17
La cosmographie universelle d'André Thevet. Quatrième partie du monde / ,... ill     15
[Illustrations de Cosmographie universelle] / [Non identifié] ; André Thevet, au 

## 4. Export PDF — correspondances Mayence 1545 → Gallica

Une page par illustration — image source (Mayence 1545) + meilleur résultat Gallica + métadonnées.

In [4]:
# Meilleur résultat Gallica par illustration
df_top = (df_mayence
          .sort_values("score", ascending=False)
          .groupby("mayence_image")
          .first()
          .reset_index()
          .sort_values("score", ascending=False))

print(f"✓ {len(df_top)} illustrations à exporter")

with PdfPages(CHEMIN_PDF) as pdf:
    for i, (_, row) in enumerate(df_top.iterrows()):
        print(f"  {i+1}/{len(df_top)}...", end="\r")

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.patch.set_facecolor("#f8f5f0")
        fig.suptitle(
            f"Illustration {i+1}/{len(df_top)} — Mayence 1545 → Gallica",
            fontsize=11, fontweight="bold"
        )

        # Colonne 1 — Image source Mayence
        ax_src = axes[0]
        chemin = f"{DOSSIER_MAYENCE}/{row['mayence_image']}"
        if os.path.exists(chemin):
            ax_src.imshow(Image.open(chemin).convert("RGB"))
        ax_src.set_title(f"Source — Mayence 1545\n{row['mayence_image'][:35]}",
                         fontsize=8, fontweight="bold", pad=8)
        ax_src.axis("off")

        # Colonne 2 — Meilleur résultat Gallica
        ax_res = axes[1]
        img_res = get_image_url(row["link"])
        if img_res:
            ax_res.imshow(img_res)
        score_color = "#2e7d32" if row["score"] >= 0.95 else \
                      "#f57c00" if row["score"] >= 0.93 else "#b71c1c"
        ax_res.set_title(
            f"Meilleur résultat Gallica — Score : {row['score']}",
            fontsize=8, color=score_color, fontweight="bold", pad=8
        )
        ax_res.axis("off")

        # Colonne 3 — Métadonnées
        ax_info = axes[2]
        ax_info.axis("off")
        info = (
            f"Titre :\n{str(row['titre'])}\n\n"
            f"Auteur :\n{str(row['auteur'])}\n\n"
            f"Date : {str(row['date'])}\n\n"
            f"Corpus :\n{str(row['corpus'])}\n\n"
            f"Technique : {str(row['technique'])}\n\n"
            f"Genre :\n{str(row['genre'])}\n\n"
            f"Lien :\n{str(row['link'])[:60]}"
        )
        ax_info.text(0.05, 0.95, info,
                     transform=ax_info.transAxes,
                     fontsize=7.5, va="top", ha="left",
                     bbox=dict(boxstyle="round", facecolor="white",
                               alpha=0.9, edgecolor="#cccccc"))

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"\n✓ PDF généré : {CHEMIN_PDF}")
print(f"  {len(df_top)} pages — une illustration par page")

✓ 52 illustrations à exporter
  52/52...
✓ PDF généré : ../../resultats/similarite/correspondances_mayence1545_gallica.pdf
  52 pages — une illustration par page
